# LMA Pretraining — Model H (Hindi) & Model L (Nepali)

Trains both monolingual GPT models from `report/phase2_model_report.md` (tied embedding,
pre-norm, RoPE, 16k vocab, d_model=384, n_layers=11, ~25.66M params each), on **GPU T4 x2**,
one model per GPU, concurrently — the two are fully independent so there's no reason to
run them sequentially when two GPUs are available.

**Before running:**

1. **Add these two datasets** (Add Input):

   | Dataset | Contents |
   | --- | --- |
   | `sohailmemon/lma-individual` | `hindi/{train,val,test}.txt`, `hindi/stats.json`, same for `nepali/` — optionally also `tokenized/hindi/{train,val,test}.bin` + `tokenized/hindi/tokenized_meta.json` (from `tokenized.zip`) to skip re-tokenizing |
   | `sohailmemon/lma-individual-files` | `hindi/tokenizer/`, `nepali/tokenizer/`, `scripts/training/*.py` |

2. **Set accelerator**: Notebook settings -> Accelerator -> **GPU T4 x2**.
3. **Add two Kaggle secrets** (Add-ons -> Secrets): `WANDB_API_KEY` (from
   [wandb.ai/authorize](https://wandb.ai/authorize)) and `HF_TOKEN` (a HF token with
   *write* access, from huggingface.co/settings/tokens).

Every checkpoint is pushed to both **Weights & Biases** (loss/lr curves, downloadable
anytime from your W&B dashboard) and the **Hugging Face Hub** (`best.pt`, `last.pt`,
`config.json`, private repos, re-pushed on every checkpoint interval — including
`last.pt`, so the most recent state is always recoverable even if the Kaggle session
dies mid-run). **If a run crashes or the session times out, just re-run this whole
notebook from the top** — it checks the HF Hub for an existing checkpoint before
training and resumes from it automatically (§1); see §5 for the details and how to
override it.


## 0. Setup

In [ ]:
!pip install -q sentencepiece wandb huggingface_hub


In [ ]:
import shutil
from pathlib import Path

FILES_DS = Path("/kaggle/input/datasets/sohailmemon/lma-individual-files")
DATA_DS = Path("/kaggle/input/datasets/sohailmemon/lma-individual")
WORK = Path("/kaggle/working")

# scripts.training.* does package-relative imports, so the "scripts" folder needs to
# live at the top of a writable, importable location — copy it out of the read-only
# /kaggle/input mount into /kaggle/working.
if (WORK / "scripts").exists():
    shutil.rmtree(WORK / "scripts")
shutil.copytree(FILES_DS / "scripts", WORK / "scripts")

import sys
sys.path.insert(0, str(WORK))

print("scripts copied to:", WORK / "scripts")


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
n_gpu = torch.cuda.device_count()
print("gpu count:", n_gpu)
for i in range(n_gpu):
    print(f"  cuda:{i}", torch.cuda.get_device_name(i))

assert n_gpu >= 2, (
    "this notebook pins Hindi to cuda:0 and Nepali to cuda:1 — set the accelerator to "
    "GPU T4 x2 in notebook settings, or fall back to running scripts.training.train "
    "sequentially by hand for each language if only one GPU is available."
)


### Logging secrets

Reads `WANDB_API_KEY` and `HF_TOKEN` from Kaggle Secrets (Add-ons -> Secrets) into the
environment. `train.py` picks both up automatically (wandb via `WANDB_API_KEY`,
`huggingface_hub` via `HF_TOKEN`) — no key ever needs to be pasted into this notebook.


In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    print("loaded WANDB_API_KEY and HF_TOKEN from Kaggle secrets")
except Exception as e:
    print(f"could not load Kaggle secrets ({e}) — relying on already-set env vars instead")

assert os.environ.get("WANDB_API_KEY"), "add the WANDB_API_KEY Kaggle secret (Add-ons -> Secrets)"
assert os.environ.get("HF_TOKEN"), "add the HF_TOKEN Kaggle secret (Add-ons -> Secrets)"


## 1. Config

Everything you're likely to want to tune lives here. `EPOCHS` controls how many passes
over each language's real token count `--max-iters` is set to (computed after
tokenizing, in §2) — the actual number of gradient steps, not a guess.

`BATCH_SIZE`/`GRAD_ACCUM_STEPS` are split the way they are because a T4 has 16GB: at
this model size, a batch of 64 run in one shot OOMs on a T4 during backward (activation
memory, not parameter memory, dominates) — running it as 4 accumulated micro-batches of
16 keeps the same effective batch size and gradient statistics at a quarter of the peak
memory. If you switch to a bigger GPU, raising `BATCH_SIZE` and lowering
`GRAD_ACCUM_STEPS` to match trains faster for the same effective batch.

Repos are created (private, by default) under `HF_USERNAME` (`sohailtsm`) on first run:
`huggingface.co/sohailtsm/lma-hindi-gpt` and `.../lma-nepali-gpt`.


In [ ]:
LANGS = ["hindi", "nepali"]

BATCH_SIZE = 16           # micro-batch — this is what actually sits in GPU memory at once
GRAD_ACCUM_STEPS = 4      # effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS = 64
BLOCK_SIZE = 512          # = T in the report; matches RoPE's precomputed table
EPOCHS = 2                # passes over each language's full token stream

WARMUP_ITERS = 300
LR = 3e-4
MIN_LR = 3e-5
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

EVAL_INTERVAL = 500
EVAL_ITERS = 100
LOG_INTERVAL = 50
CHECKPOINT_INTERVAL = 1000

WANDB_PROJECT = "lma-individual-pretrain"

HF_USERNAME = "sohailtsm"
HF_REPOS = {lang: f"{HF_USERNAME}/lma-{lang}-gpt" for lang in LANGS}
HF_PUBLIC = False

# Auto-filled below by checking the HF Hub repos for an existing last.pt — leave as
# None here. Force a from-scratch run for a language by setting its entry to False
# instead of None after that cell runs.
RESUME = {"hindi": None, "nepali": None}


### Auto-resume from the Hugging Face Hub

Since every checkpoint interval pushes `last.pt` to `HF_REPOS[lang]`, re-running this
whole notebook from the top after a crash or a Kaggle session timeout picks up training
where it left off automatically — no need to notice a run failed and manually point
`RESUME` at anything. First run (no checkpoint pushed yet) trains from scratch, same as
before.


In [ ]:
from huggingface_hub import hf_hub_download

for lang in LANGS:
    if RESUME[lang] is False:
        continue
    try:
        RESUME[lang] = hf_hub_download(HF_REPOS[lang], "last.pt")
        print(f"[{lang}] found existing checkpoint on {HF_REPOS[lang]}, will resume from it")
    except Exception as e:
        print(f"[{lang}] no existing checkpoint on {HF_REPOS[lang]} ({type(e).__name__}); "
              f"starting from scratch")


## 2. Tokenize each language into a flat token stream

`prepare_bin.py` reads `train.txt`/`val.txt`/`test.txt` and writes `{split}.bin` — a flat
`uint16` array of token ids with `<eos>` between documents — plus `tokenized_meta.json`
with the real token counts, which §3 uses to size `--max-iters`.

If `lma-individual` already has `tokenized/<lang>/{train,val,test}.bin` and
`tokenized/<lang>/tokenized_meta.json` (from uploading `tokenized.zip`), that
pre-tokenized data is used directly from the read-only mount instead — tokenizing ~930M
tokens/language takes a few minutes of CPU time that's cheap to skip once it's already
been done once.


In [ ]:
import subprocess

BIN_DIR = {}

for lang in LANGS:
    pretokenized = DATA_DS / "tokenized" / lang
    if (pretokenized / "train.bin").exists() and (pretokenized / "tokenized_meta.json").exists():
        print(f"[{lang}] found pre-tokenized data in {pretokenized}, skipping prepare_bin.py")
        BIN_DIR[lang] = pretokenized
        continue

    out_dir = WORK / lang / "data"
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python", "-u", "-m", "scripts.training.prepare_bin",
        "--lang", lang,
        "--data-dir", str(DATA_DS / lang),
        "--tokenizer", str(FILES_DS / lang / "tokenizer" / f"{lang}_unigram_16k.model"),
        "--out-dir", str(out_dir),
    ]
    print(f"=== tokenizing {lang} ===")
    subprocess.run(cmd, cwd=WORK, check=True)
    BIN_DIR[lang] = out_dir


In [ ]:
import json

max_iters = {}
for lang in LANGS:
    meta = json.loads((BIN_DIR[lang] / "tokenized_meta.json").read_text())
    train_tokens = meta["tokens"]["train"]
    tokens_per_iter = BATCH_SIZE * GRAD_ACCUM_STEPS * BLOCK_SIZE
    iters = max(1, (train_tokens * EPOCHS) // tokens_per_iter)
    max_iters[lang] = iters
    print(f"{lang}: {train_tokens:,} train tokens -> {iters:,} iters for {EPOCHS} epoch(s) "
          f"({tokens_per_iter:,} tokens/iter)")


## 3. Pretrain both models in parallel (GPU T4 x2)

Hindi runs on `cuda:0`, Nepali on `cuda:1`, as two subprocesses, each training at the
micro-batch/grad-accumulation split from §1 (`BATCH_SIZE` x `GRAD_ACCUM_STEPS`) — needed
because this model OOMs a T4 during backward at a plain batch of 64. Each subprocess
streams its own stderr (loss/lr per iter) to a log file; the cell below tails both
interleaved with a `[lang]` prefix until both finish. Live loss/lr curves are on your
W&B project dashboard
(`WANDB_PROJECT`) the moment training starts — no need to wait for this cell to print
anything to watch progress there. Checkpoints land in
`/kaggle/working/<lang>/checkpoints/{best,last}.pt` and are pushed to
`HF_REPOS[lang]` on every checkpoint interval.


In [ ]:
def train_async(lang: str, device: str, log_path: Path) -> subprocess.Popen:
    out_dir = WORK / lang / "checkpoints"
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python", "-u", "-m", "scripts.training.train",
        "--lang", lang,
        "--data-dir", str(BIN_DIR[lang]),
        "--out-dir", str(out_dir),
        "--device", device,
        "--batch-size", str(BATCH_SIZE),
        "--grad-accum-steps", str(GRAD_ACCUM_STEPS),
        "--block-size", str(BLOCK_SIZE),
        "--max-iters", str(max_iters[lang]),
        "--warmup-iters", str(WARMUP_ITERS),
        "--lr", str(LR),
        "--min-lr", str(MIN_LR),
        "--weight-decay", str(WEIGHT_DECAY),
        "--grad-clip", str(GRAD_CLIP),
        "--eval-interval", str(EVAL_INTERVAL),
        "--eval-iters", str(EVAL_ITERS),
        "--log-interval", str(LOG_INTERVAL),
        "--checkpoint-interval", str(CHECKPOINT_INTERVAL),
        "--wandb-project", WANDB_PROJECT,
        "--wandb-run-name", f"{lang}-gpt",
        "--hf-repo-id", HF_REPOS[lang],
    ]
    if HF_PUBLIC:
        cmd.append("--hf-public")
    if RESUME.get(lang):
        cmd += ["--resume", RESUME[lang]]
    log_f = open(log_path, "w")
    env = os.environ | {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    return subprocess.Popen(cmd, cwd=WORK, stdout=log_f, stderr=subprocess.STDOUT, env=env)


def stream_until_done(jobs: dict[str, tuple[subprocess.Popen, Path]], poll_s: float = 5.0) -> None:
    positions = {lang: 0 for lang in jobs}
    while any(p.poll() is None for p, _ in jobs.values()):
        for lang, (p, log_path) in jobs.items():
            with open(log_path) as f:
                f.seek(positions[lang])
                new_text = f.read()
                positions[lang] = f.tell()
            for line in new_text.splitlines():
                print(f"[{lang}] {line}")
        time.sleep(poll_s)
    for lang, (p, log_path) in jobs.items():
        with open(log_path) as f:
            f.seek(positions[lang])
            new_text = f.read()
        for line in new_text.splitlines():
            print(f"[{lang}] {line}")
        print(f"[{lang}] process exited with code {p.returncode}")


In [ ]:
import time

jobs = {
    "hindi": (train_async("hindi", "cuda:0", WORK / "hindi_train.log"), WORK / "hindi_train.log"),
    "nepali": (train_async("nepali", "cuda:1", WORK / "nepali_train.log"), WORK / "nepali_train.log"),
}
stream_until_done(jobs)

for lang, (p, _) in jobs.items():
    if p.returncode != 0:
        raise RuntimeError(f"{lang} training failed with exit code {p.returncode} — see "
                            f"{WORK / (lang + '_train.log')}")


## 4. Where everything ended up

- **Checkpoints**: `/kaggle/working/<lang>/checkpoints/{best,last}.pt` — kept when you
  *Save Version*.
- **Weights & Biases**: `https://wandb.ai/<your-entity>/lma-individual-pretrain` — one run
  per language (`hindi-gpt`, `nepali-gpt`), with train/val loss, lr, and ms/iter logged
  throughout. Downloadable from there anytime (CSV export or the `wandb` API), independent
  of whether this Kaggle session is still alive.
- **Hugging Face Hub**: `https://huggingface.co/<HF_USERNAME>/lma-hindi-gpt` and
  `.../lma-nepali-gpt` (private) — `best.pt`, `last.pt`, `config.json`, re-uploaded on every
  checkpoint interval during training, not just at the end.


## 5. Resuming after a session ends

Kaggle sessions have a hard time limit. **Default behavior needs no action from you**:
§1's auto-resume cell checks `HF_REPOS[lang]` for a `last.pt` on every run, so just
re-running this same notebook from the top after a crash or timeout picks up where the
last checkpoint left off.

Two situations where you'd override that:

- **Force a from-scratch run** for one language (ignore any existing HF checkpoint):
  set `RESUME["hindi"] = False` in §1 before running the auto-resume cell.
- **Resume from this notebook's own `/kaggle/working` output instead of HF** (e.g. HF
  push failed for some reason but a local checkpoint exists): **Save Version**, add that
  version's output as a new input dataset (**Add Input -> Notebook Output Files**), then
  set `RESUME["hindi"] = "/kaggle/input/<previous-output>/hindi/checkpoints/last.pt"`
  *after* the auto-resume cell runs (so it isn't overwritten).


## 6. Summary

In [ ]:
for lang in LANGS:
    ckpt_dir = WORK / lang / "checkpoints"
    print(f"--- {lang} ---")
    if not ckpt_dir.exists():
        print("  no checkpoints yet")
        continue
    for f in sorted(ckpt_dir.iterdir()):
        print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")
    print(f"  HF repo: https://huggingface.co/{HF_REPOS[lang]}")
